# Quickstart: place one ground station with SCORE

This notebook runs SCORE (`problem.method=nelder_ccgs`) for a single ground station serving two
CAPELLA satellites over a one-day window, then scores the result and plots it on a map. It takes
a few minutes on a laptop.

The paper's runs use a seven-day window and more stations and satellites; `scripts/run_sweeps.sh`
launches those sweeps through `src/main.py`.

In [ ]:
import sys
import tempfile
from pathlib import Path

# Make src/ importable whether the notebook runs from examples/ or the repository root
REPO = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

from hydra import compose, initialize_config_dir

import main as gsopt  # loads the land and city boundary shapefiles
from methods.free_select.scipy_ccgs import nelder_mead_scipy_ccgs
from common.plotting import plot_img

## Configure the run

`config/config.yaml` holds the defaults. Overrides use the same `key=value` syntax as the command
line, e.g. `python src/main.py problem.gs_num=3`.

In [ ]:
with initialize_config_dir(config_dir=str(REPO / "config"), version_base=None):
    cfg = compose(config_name="config", overrides=[
        "problem.method=nelder_ccgs",
        "problem.gs_num=1",
        "problem.sat_num=2",
        "scenario.constellations=CAPELLA",
        "end_epoch.day=2",        # one-day window (the default is seven days)
        "experiments.ccgs=1",     # one SCORE pass over the stations
        "debug.verbose=False",
    ])

## Build and propagate the satellites

In [ ]:
epc_start, epc_end, satellites = gsopt.setup_scenario(cfg)
print(f"{len(satellites)} satellites from {epc_start} to {epc_end}")

## Optimize the station location

SciPy may warn that Nelder-Mead reached its iteration limit; each search is capped at 100 iterations.

In [ ]:
gs_list, gs_list_plot = nelder_mead_scipy_ccgs(
    cfg, gsopt.land_data, epc_start, epc_end, satellites, gsopt.eval_counter, gsopt.city_data)

## Score the solution

Contacts are scored after contact exclusion: no overlapping contacts per satellite or station, and
contacts of 180 s or less are dropped. `data_downlink` is in bits.

In [ ]:
results = gsopt.summarize_free(cfg, satellites, gs_list, gs_list_plot, epc_start, epc_end)
for key, value in results.items():
    print(f"{key:>14}: {value}")

## Plot the station and its coverage

The shaded circle is the area that sees a satellite at 570 km altitude above 10 degrees elevation.

In [ ]:
fig = plot_img(gs_list_plot, str(Path(tempfile.gettempdir()) / "quickstart_stations.png"))